# Snapshot download with progress

> Wrap `huggingface_hub.snapshot_download` so per-file progress flows to the plugin's `report_progress`, defeating the substrate's stall detector during downloads.

In [ ]:
#| default_exp download

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path
from typing import Callable, Optional

from huggingface_hub import snapshot_download
from tqdm.auto import tqdm as _tqdm

The download phase is where HF Hub plugins spend their cold-start time, and it is
silent enough to trip the substrate's prefetch stall detector. This helper hooks
`snapshot_download`'s `tqdm_class` so each progress tick calls
`report_progress(fraction, "downloading <file>")`, advancing the (progress, message)
tuple the substrate polls.

Pair it with `self.heartbeat(...)` around the subsequent
`from_pretrained(..., local_files_only=True)`: download progress comes from this
helper, and the heartbeat covers the silent model-construction phase that follows.

Returns the local snapshot directory. LavaSR-style constructors that take a path
can pass the returned `Path` to bypass their internal (progress-less) download.

In [ ]:
#| export
def snapshot_download_with_progress(
    repo_id: str,                                                    # HF Hub repo id, e.g. "mistralai/Voxtral-Mini-3B-2507"
    *,
    report_progress: Optional[Callable[[float, str], None]] = None,  # Plugin's report_progress(fraction, message)
    cache_dir: Optional[str] = None,                                 # HF cache override (HFCacheConfig.cache_dir)
    revision: Optional[str] = None,                                  # Git rev / tag / commit to pin
    local_files_only: bool = False,                                  # Use cached files only; no network
    **kwargs,                                                        # Forwarded to huggingface_hub.snapshot_download
) -> Path:                                                           # Local path to the downloaded snapshot
    """Download an HF Hub snapshot, streaming per-file progress to `report_progress`.

    When `report_progress` is given, a `tqdm_class` subclass forwards each update
    as `report_progress(downloaded / total, "downloading <file>")`. When it is
    None, the default HF Hub progress bars are used unchanged.

    Returns the local snapshot directory; subsequent `from_pretrained` calls with
    the same `cache_dir` + `local_files_only=True` hit the populated cache.
    """
    tqdm_class = None
    if report_progress is not None:
        class _ReportingTqdm(_tqdm):
            def update(self, n=1):
                super().update(n)
                if self.total:
                    report_progress(self.n / self.total, f"downloading {self.desc or repo_id}")
        tqdm_class = _ReportingTqdm
    return Path(snapshot_download(
        repo_id,
        cache_dir=cache_dir,
        revision=revision,
        local_files_only=local_files_only,
        tqdm_class=tqdm_class,
        **kwargs,
    ))

Tests. (Network-free: `local_files_only=True` on an uncached repo fails fast.)

In [ ]:
# local_files_only=True on a not-cached repo fails fast without network, proving
# our kwargs forward through to snapshot_download.
_BOGUS = "cjm-nonexistent-org/definitely-not-a-real-repo-xyz"
_raised = False
try:
    snapshot_download_with_progress(_BOGUS, local_files_only=True)
except Exception:
    _raised = True
assert _raised, "expected a local-cache-miss error with local_files_only=True"

# Passing a report_progress callback is accepted (the tqdm subclass builds cleanly);
# same fast-fail path, no crash constructing the reporting tqdm.
_calls = []
try:
    snapshot_download_with_progress(
        _BOGUS, local_files_only=True,
        report_progress=lambda f, m: _calls.append((f, m)),
    )
except Exception:
    pass